# 01 — OGG Data Exploration

## Goal
Build the first analysis-ready dataset for **Kahului Airport (OGG), Maui** by combining historical domestic flight operations with airport weather and severe-weather incident context.

### Primary sources
- **BTS Reporting Carrier On-Time Performance** — individual domestic flights, schedules, actual times, delays, cancellations, diversions, and delay causes.
- **NOAA/NCEI Local Climatological Data (LCD)** — hourly airport weather observations.
- **NOAA Storm Events Database** — severe-weather incident metadata and narratives.

### Unit of analysis
For the baseline model, **one row = one scheduled flight departing from or arriving at OGG**.


## Initial prediction target

We will derive a categorical target:

- `normal`: completed flight with < 15 min arrival delay
- `delay`: completed flight with 15–179 min arrival delay
- `severe_delay`: completed flight with >= 180 min arrival delay
- `cancelled`: cancelled flight

Diversions will initially be retained as a separate operational flag and can later become a fifth class.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robustly locate the repository root whether Jupyter starts in the
# repository root or inside the notebooks/ directory.
cwd = Path.cwd().resolve()
if (cwd / 'data').exists() and (cwd / 'notebooks').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists() and (cwd.parent / 'notebooks').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        f"Could not locate repository root from working directory: {cwd}"
    )

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
INCIDENT_DIR = RAW_DIR / 'incidents'

print(f'Working directory: {cwd}')
print(f'Repository root:   {ROOT}')
print(f'Raw data dir:      {RAW_DIR}')
print(f'Raw data exists:   {RAW_DIR.exists()}')

RAW_DIR, PROCESSED_DIR, INCIDENT_DIR


Working directory: /workspaces/flight-disruption-ai/notebooks
Repository root:   /workspaces/flight-disruption-ai
Raw data dir:      /workspaces/flight-disruption-ai/data/raw
Raw data exists:   True


(PosixPath('/workspaces/flight-disruption-ai/data/raw'),
 PosixPath('/workspaces/flight-disruption-ai/data/processed'),
 PosixPath('/workspaces/flight-disruption-ai/data/raw/incidents'))

## Expected BTS fields

We will keep the smallest useful subset first:

`FlightDate`, `Reporting_Airline`, `Flight_Number_Reporting_Airline`, `Origin`, `Dest`, `CRSDepTime`, `DepTime`, `DepDelay`, `CRSArrTime`, `ArrTime`, `ArrDelay`, `Cancelled`, `CancellationCode`, `Diverted`, `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`, `Distance`.


In [2]:
BTS_KEEP = [
    'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline',
    'Origin', 'Dest', 'CRSDepTime', 'DepTime', 'DepDelay',
    'CRSArrTime', 'ArrTime', 'ArrDelay', 'Cancelled', 'CancellationCode',
    'Diverted', 'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay', 'Distance'
]
BTS_KEEP


['FlightDate',
 'Reporting_Airline',
 'Flight_Number_Reporting_Airline',
 'Origin',
 'Dest',
 'CRSDepTime',
 'DepTime',
 'DepDelay',
 'CRSArrTime',
 'ArrTime',
 'ArrDelay',
 'Cancelled',
 'CancellationCode',
 'Diverted',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityDelay',
 'LateAircraftDelay',
 'Distance']

## Load downloaded BTS files

Place BTS CSV files in `data/raw/bts/`. We will start with a manageable recent historical window and expand after validating the pipeline.


In [3]:
bts_dir = RAW_DIR / 'bts'
print(f'Looking for BTS files in: {bts_dir.resolve()}')
print(f'BTS directory exists: {bts_dir.exists()}')

bts_files = sorted(list(bts_dir.glob('*.csv')) + list(bts_dir.glob('*.csv.gz')))
print(f'Found {len(bts_files)} BTS files')
bts_files[:5]


Looking for BTS files in: /workspaces/flight-disruption-ai/data/raw/bts
BTS directory exists: True
Found 1 BTS files


[PosixPath('/workspaces/flight-disruption-ai/data/raw/bts/ogg_flights_2020_2026.csv.gz')]

In [4]:
def load_bts_ogg(files):
    frames = []
    for path in files:
        df = pd.read_csv(path, low_memory=False)
        available = [c for c in BTS_KEEP if c in df.columns]
        df = df[available].copy()
        if {'Origin', 'Dest'}.issubset(df.columns):
            df = df[(df['Origin'] == 'OGG') | (df['Dest'] == 'OGG')]
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

flights = load_bts_ogg(bts_files)
flights.shape


(330615, 18)

In [5]:
def assign_disruption_class(row):
    if row.get('Cancelled', 0) == 1:
        return 'cancelled'
    delay = row.get('ArrDelay', np.nan)
    if pd.isna(delay):
        return 'unknown'
    if delay < 15:
        return 'normal'
    if delay < 180:
        return 'delay'
    return 'severe_delay'

if not flights.empty:
    flights['disruption_class'] = flights.apply(assign_disruption_class, axis=1)
    display(flights['disruption_class'].value_counts(dropna=False))


disruption_class
unknown      326198
cancelled      4417
Name: count, dtype: int64

## Expected NOAA LCD weather fields

At minimum we want timestamp-aligned measures for temperature, dew point, relative humidity, station pressure, visibility, wind speed/direction/gusts, precipitation, cloud/sky condition, and weather type. Exact LCD column names can vary by product/version, so the next step is schema inspection after downloading the first OGG station file.


In [6]:
weather_dir = RAW_DIR / 'weather'
print(f'Looking for weather files in: {weather_dir.resolve()}')
print(f'Weather directory exists: {weather_dir.exists()}')

weather_files = sorted(list(weather_dir.glob('*.csv')) + list(weather_dir.glob('*.csv.gz')))
print(f'Found {len(weather_files)} weather files')

if weather_files:
    weather_sample = pd.read_csv(weather_files[0], low_memory=False)
    print(weather_sample.shape)
    display(pd.DataFrame({'column': weather_sample.columns}))


Looking for weather files in: /workspaces/flight-disruption-ai/data/raw/weather
Weather directory exists: True
Found 1 weather files
(66074, 125)


,column
0,STATION
1,DATE
2,LATITUDE
3,LONGITUDE
4,ELEVATION
...,...
120,BackupEquipment
121,BackupLatitude
122,BackupLongitude
123,BackupName


## Load NOAA Hawaii Storm Events

The downloader stores the filtered Hawaii Storm Events file under `data/raw/incidents/`. This dataset provides event-level context such as event type, timing, affected area, magnitude (when available), coordinates, damage/injury fields, and event narratives.


In [7]:
print(f'Looking for storm-event files in: {INCIDENT_DIR.resolve()}')
print(f'Storm-event directory exists: {INCIDENT_DIR.exists()}')

storm_files = sorted(list(INCIDENT_DIR.glob('*.csv')) + list(INCIDENT_DIR.glob('*.csv.gz')))
print(f'Found {len(storm_files)} storm-event files')

if storm_files:
    storms = pd.read_csv(storm_files[0], low_memory=False)
    print(storms.shape)
    display(storms.head())
    display(pd.DataFrame({'column': storms.columns}))
else:
    storms = pd.DataFrame()


Looking for storm-event files in: /workspaces/flight-disruption-ai/data/raw/incidents
Storm-event directory exists: True
Found 1 storm-event files
(2613, 51)


,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
0,202003,20,1900,202003,22,1400,145695,875114,HAWAII,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A rather modest northwest swell produced surf ...,NaN,CSV
1,202003,20,1900,202003,22,1400,145695,875115,HAWAII,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A rather modest northwest swell produced surf ...,NaN,CSV
2,202003,24,400,202003,25,1900,145698,875119,HAWAII,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The combination of a swell from the southern h...,NaN,CSV
3,202003,24,400,202003,28,1900,145698,875120,HAWAII,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The combination of a swell from the southern h...,NaN,CSV
4,202003,16,2330,202003,17,1539,145684,875053,HAWAII,15,...,0.83,S,PRINCEVILLE,22.2095,-159.4815,22.2102,-159.4797,"An upper low, known as a kona low in Hawaii, a...",Kuhio Highway near the Hanalei Bridge became i...,CSV


,column
0,BEGIN_YEARMONTH
1,BEGIN_DAY
2,BEGIN_TIME
3,END_YEARMONTH
4,END_DAY
5,END_TIME
6,EPISODE_ID
7,EVENT_ID
8,STATE
9,STATE_FIPS


## Join strategy

1. Convert scheduled flight time at OGG into a proper Hawaii-local timestamp.
2. Normalize NOAA weather timestamps.
3. For each flight, match the nearest prior weather observation (or aggregate over windows such as 1h, 3h, and 6h before scheduled departure/arrival).
4. Add daily/event-level severe-weather indicators from NOAA Storm Events.
5. Later add airline-, airport-, congestion-, and inbound-aircraft features.


## Phase 1 validation checklist

- Confirm OGG flight file loads from `data/raw/bts/`
- Confirm OGG NOAA LCD weather file loads from `data/raw/weather/`
- Confirm Hawaii Storm Events file loads from `data/raw/incidents/`
- Verify timestamps and Hawaii time-zone handling
- Inspect missingness and cancellation coding
- Build one merged flight-weather-event table
- Plot disruption rate against wind, visibility, precipitation, and severe-weather context

After these checks pass, begin baseline modeling and recovery-time analysis.
